# 03 - Frequency Sweep and Insertion Loss Measurement

This is the next logical notebook after CW power verification.

Instead of sweeping **power at one frequency**, this notebook sweeps **frequency at a fixed power** and measures **how much signal is delivered through the RF path**.

That makes it a natural introduction to a common RF lab measurement:

**insertion loss versus frequency**.

## What this notebook covers

- Fixed-power CW generation
- Frequency sweep across a band
- Spectrum analyzer marker measurements
- Insertion loss calculation
- Frequency-response plots
- Mock mode for offline development

## Measurement definition

At each sweep point, insertion loss is computed as:

$\mathrm{Insertion\ Loss\ (dB)} = P_{\mathrm{gen}} - P_{\mathrm{meas}}$

In [ ]:
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

In [ ]:
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print('Discovered VISA resources:')
pprint(resources)

In [ ]:
def configure_common_session(inst, timeout_ms=5000):
    inst.timeout = timeout_ms
    inst.write_termination = '\n'
    inst.read_termination = '\n'
    return inst

def identify(inst):
    return inst.query('*IDN?').strip()

def safe_close(inst):
    try:
        inst.close()
    except Exception:
        pass

In [ ]:
class SignalGenerator:
    def __init__(self, inst):
        self.inst = inst

    def idn(self):
        return identify(self.inst)

    def reset(self):
        self.inst.write('*RST')

    def set_frequency_hz(self, freq_hz):
        self.inst.write(f'FREQ {freq_hz}')

    def get_frequency_hz(self):
        return float(self.inst.query('FREQ?'))

    def set_power_dbm(self, power_dbm):
        self.inst.write(f'POW {power_dbm}DBM')

    def get_power_dbm(self):
        return float(self.inst.query('POW?'))

    def output_on(self):
        self.inst.write('OUTP ON')

    def output_off(self):
        self.inst.write('OUTP OFF')


class SpectrumAnalyzer:
    def __init__(self, inst):
        self.inst = inst

    def idn(self):
        return identify(self.inst)

    def reset(self):
        self.inst.write('*RST')

    def set_center_frequency_hz(self, freq_hz):
        self.inst.write(f'FREQ:CENT {freq_hz}')

    def set_span_hz(self, span_hz):
        self.inst.write(f'FREQ:SPAN {span_hz}')

    def set_rbw_hz(self, rbw_hz):
        self.inst.write(f'BAND {rbw_hz}')

    def place_marker_at_peak(self):
        self.inst.write('CALC:MARK1:MAX')

    def marker_frequency_hz(self):
        return float(self.inst.query('CALC:MARK1:X?'))

    def marker_power_dbm(self):
        return float(self.inst.query('CALC:MARK1:Y?'))

In [ ]:
SIGGEN_RESOURCE = None
SA_RESOURCE = None

# Example:
# SIGGEN_RESOURCE = 'TCPIP0::192.168.0.55::inst0::INSTR'
# SA_RESOURCE = 'TCPIP0::192.168.0.56::inst0::INSTR'

In [ ]:
class MockSignalGeneratorInstrument:
    def __init__(self):
        self.timeout = 5000
        self.write_termination = '\n'
        self.read_termination = '\n'
        self.frequency_hz = 1e9
        self.power_dbm = -20.0
        self.output_enabled = False

    def write(self, cmd):
        cmd = cmd.strip().upper()
        if cmd == '*RST':
            self.frequency_hz = 1e9
            self.power_dbm = -20.0
            self.output_enabled = False
        elif cmd.startswith('FREQ '):
            self.frequency_hz = float(cmd.split()[1])
        elif cmd.startswith('POW '):
            value = cmd.split()[1].replace('DBM', '')
            self.power_dbm = float(value)
        elif cmd == 'OUTP ON':
            self.output_enabled = True
        elif cmd == 'OUTP OFF':
            self.output_enabled = False

    def query(self, cmd):
        cmd = cmd.strip().upper()
        if cmd == '*IDN?':
            return 'MOCK,SIGGEN,1001,1.0'
        if cmd == 'FREQ?':
            return str(self.frequency_hz)
        if cmd == 'POW?':
            return str(self.power_dbm)
        return '0'

    def close(self):
        pass


class MockSpectrumAnalyzerInstrument:
    def __init__(self, siggen_source):
        self.timeout = 5000
        self.write_termination = '\n'
        self.read_termination = '\n'
        self.siggen_source = siggen_source
        self.center_hz = 1e9
        self.span_hz = 1e6
        self.rbw_hz = 1e3

    def modeled_path_loss_db(self, freq_hz):
        freq_ghz = freq_hz / 1e9
        baseline = 1.0 + 0.8 * (freq_ghz - 1.0)
        ripple = 0.25 * np.sin(2 * np.pi * (freq_ghz - 1.0) * 1.6)
        notch = 0.0
        if 2.15e9 <= freq_hz <= 2.35e9:
            notch = 1.2 * np.exp(-((freq_hz - 2.25e9) ** 2) / (2 * (0.06e9) ** 2))
        return baseline + ripple + notch

    def write(self, cmd):
        cmd_u = cmd.strip().upper()
        if cmd_u == '*RST':
            self.center_hz = 1e9
            self.span_hz = 1e6
            self.rbw_hz = 1e3
        elif cmd_u.startswith('FREQ:CENT '):
            self.center_hz = float(cmd.split()[1])
        elif cmd_u.startswith('FREQ:SPAN '):
            self.span_hz = float(cmd.split()[1])
        elif cmd_u.startswith('BAND '):
            self.rbw_hz = float(cmd.split()[1])

    def query(self, cmd):
        cmd_u = cmd.strip().upper()
        if cmd_u == '*IDN?':
            return 'MOCK,SPECTRUM-ANALYZER,2001,1.0'
        if cmd_u == 'CALC:MARK1:X?':
            return str(self.siggen_source.frequency_hz)
        if cmd_u == 'CALC:MARK1:Y?':
            if not self.siggen_source.output_enabled:
                return '-120.0'
            loss = self.modeled_path_loss_db(self.siggen_source.frequency_hz)
            measured = self.siggen_source.power_dbm - loss
            return str(measured)
        return '0'

    def close(self):
        pass

In [ ]:
siggen = None
sa = None
real_hardware_connected = False

if SIGGEN_RESOURCE and SA_RESOURCE:
    try:
        siggen_inst = configure_common_session(rm.open_resource(SIGGEN_RESOURCE))
        sa_inst = configure_common_session(rm.open_resource(SA_RESOURCE))
        siggen = SignalGenerator(siggen_inst)
        sa = SpectrumAnalyzer(sa_inst)
        print('Signal generator:', siggen.idn())
        print('Spectrum analyzer:', sa.idn())
        real_hardware_connected = True
    except Exception as e:
        print('Real hardware connection failed:')
        print(e)

if not real_hardware_connected:
    mock_siggen_inst = MockSignalGeneratorInstrument()
    mock_sa_inst = MockSpectrumAnalyzerInstrument(mock_siggen_inst)
    siggen = SignalGenerator(mock_siggen_inst)
    sa = SpectrumAnalyzer(mock_sa_inst)
    print('Using mock instruments for tutorial execution.')
    print('Signal generator:', siggen.idn())
    print('Spectrum analyzer:', sa.idn())

In [ ]:
start_freq_hz = 1.0e9
stop_freq_hz = 3.0e9
num_points = 41
generator_power_dbm = -20.0
analyzer_span_hz = 2.0e6
analyzer_rbw_hz = 1.0e3

freqs_hz = np.linspace(start_freq_hz, stop_freq_hz, num_points)
freqs_ghz = freqs_hz / 1e9

In [ ]:
siggen.reset()
sa.reset()

siggen.set_power_dbm(generator_power_dbm)
siggen.output_on()

measured_powers_dbm = []
insertion_loss_db = []
measured_marker_freqs_hz = []

for f_hz in freqs_hz:
    siggen.set_frequency_hz(float(f_hz))
    sa.set_center_frequency_hz(float(f_hz))
    sa.set_span_hz(analyzer_span_hz)
    sa.set_rbw_hz(analyzer_rbw_hz)
    sa.place_marker_at_peak()

    marker_freq = sa.marker_frequency_hz()
    marker_power = sa.marker_power_dbm()
    loss_db = generator_power_dbm - marker_power

    measured_marker_freqs_hz.append(marker_freq)
    measured_powers_dbm.append(marker_power)
    insertion_loss_db.append(loss_db)

measured_marker_freqs_hz = np.array(measured_marker_freqs_hz)
measured_powers_dbm = np.array(measured_powers_dbm)
insertion_loss_db = np.array(insertion_loss_db)

In [ ]:
for i in range(0, len(freqs_hz), 8):
    print(
        f'Freq: {freqs_hz[i]/1e9:5.3f} GHz | '
        f'Measured: {measured_powers_dbm[i]:6.2f} dBm | '
        f'Insertion loss: {insertion_loss_db[i]:5.2f} dB'
    )

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(freqs_ghz, measured_powers_dbm, marker='o')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Measured power (dBm)')
plt.title('Measured CW Power vs Frequency')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(freqs_ghz, insertion_loss_db, marker='o')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Insertion loss (dB)')
plt.title('Insertion Loss vs Frequency')
plt.grid(True)
plt.show()

In [ ]:
print(f'Min insertion loss:  {np.min(insertion_loss_db):.2f} dB')
print(f'Max insertion loss:  {np.max(insertion_loss_db):.2f} dB')
print(f'Mean insertion loss: {np.mean(insertion_loss_db):.2f} dB')
print(f'Ripple (pk-pk):      {np.ptp(insertion_loss_db):.2f} dB')

## Why this notebook matters

This is a realistic bridge from basic instrument control to actual RF characterization.

It sets you up well for either:

- trace acquisition and peak search, or
- gain / compression style DUT testing.

In [ ]:
try:
    siggen.output_off()
except Exception:
    pass

print('RF output disabled at end of notebook.')